# 📗 Supervised Learning — Classification (Complete Revision Notes)

Self-contained revision guide for every major classification technique.
**Dataset used:** `titanic` from seaborn (predicting `survived`).


In [ ]:
# If any of these are missing, uncomment and run:
# !pip install xgboost lightgbm imbalanced-learn scikit-learn seaborn pandas numpy --quiet


## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV, RandomizedSearchCV, learning_curve
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, LabelEncoder, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import RFE

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier, VotingClassifier, StackingClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              confusion_matrix, classification_report, roc_curve, auc,
                              precision_recall_curve, RocCurveDisplay)

import warnings
warnings.filterwarnings("ignore")

try:
    from xgboost import XGBClassifier
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False
    print("xgboost not installed -> pip install xgboost")

try:
    from lightgbm import LGBMClassifier
    LGBM_AVAILABLE = True
except ImportError:
    LGBM_AVAILABLE = False
    print("lightgbm not installed -> pip install lightgbm")

try:
    from imblearn.over_sampling import SMOTE
    SMOTE_AVAILABLE = True
except ImportError:
    SMOTE_AVAILABLE = False
    print("imbalanced-learn not installed -> pip install imbalanced-learn")

sns.set_style("whitegrid")


## 2. Load Dataset
`titanic` dataset — predict `survived` (0/1) from passenger features. Classic binary classification
problem with a mix of numeric/categorical data AND missing values, so it's great for practice.

In [ ]:
df = sns.load_dataset('titanic')
df.head()

## 3. EDA
**Why:** Understand structure, missingness, and — crucially for classification — check **class balance**,
since imbalanced classes need special handling (see Section 7).

In [ ]:
df.info()

In [ ]:
df.isnull().sum().sort_values(ascending=False)

In [ ]:
# Class balance check - important! If classes are very imbalanced, accuracy alone is misleading
plt.figure(figsize=(4,4))
sns.countplot(x='survived', data=df)
plt.title('Class Balance: Survived (0=No, 1=Yes)')
plt.show()
print(df['survived'].value_counts(normalize=True))

In [ ]:
# Correlation heatmap
plt.figure(figsize=(7,5))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
# Distribution of a key numeric feature by class
plt.figure(figsize=(6,4))
sns.kdeplot(data=df, x='age', hue='survived', fill=True)
plt.title('Age distribution by survival')
plt.show()

## 4. Feature Engineering & Missing Value Handling
**Why:** `age`, `embarked`, and `deck` have missing values. We drop `deck` (too sparse) and impute
the rest. We also engineer a `family_size` feature from `sibsp` + `parch`.

In [ ]:
data = df.drop(columns=['deck', 'embark_town', 'alive', 'class', 'who', 'adult_male'])  # drop redundant/near-duplicate columns

# Median imputation for numeric (robust to outliers, simple)
data['age'] = SimpleImputer(strategy='median').fit_transform(data[['age']])

# Mode imputation for categorical
data['embarked'] = data['embarked'].fillna(data['embarked'].mode()[0])

# Feature engineering: combine siblings/spouses + parents/children into one feature
data['family_size'] = data['sibsp'] + data['parch'] + 1
data['is_alone'] = (data['family_size'] == 1).astype(int)

data.isnull().sum()

## 5. Encoding Categorical Variables
Same logic as the regression notebook:
- **OneHot** for nominal categories used with linear/SVM/KNN models
- **LabelEncoder** works fine for tree-based models


In [ ]:
cat_cols = ['sex', 'embarked']
data_encoded = pd.get_dummies(data, columns=cat_cols, drop_first=True)
data_encoded.head()

## 6. Feature Scaling
Distance/gradient-based models (Logistic Regression, KNN, SVM) need scaled features.
Tree-based models (Decision Tree, RF, boosting) do not.

In [ ]:
num_cols = ['age', 'fare', 'family_size']
scaler_demo = StandardScaler()
scaled_demo = pd.DataFrame(scaler_demo.fit_transform(data_encoded[num_cols]), columns=[c+'_scaled' for c in num_cols])
scaled_demo.head()

## 7. Train-Test Split (Stratified) + Handling Class Imbalance
**Why stratify:** ensures both train and test sets keep the same class proportions as the full dataset —
critical for classification, especially with imbalance.

**Why SMOTE / class_weight:** If one class is much rarer, a model can get high accuracy by just
predicting the majority class every time. SMOTE creates synthetic minority-class samples;
`class_weight='balanced'` instead penalizes misclassifying the minority class more heavily.

In [ ]:
X = data_encoded.drop(columns=['survived'])
y = data_encoded['survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train class balance:\n", y_train.value_counts(normalize=True))

In [ ]:
# SMOTE demo: oversamples the minority class ONLY on the training set (never touch test set!)
if SMOTE_AVAILABLE:
    sm = SMOTE(random_state=42)
    X_train_smote, y_train_smote = sm.fit_resample(X_train_scaled, y_train)
    print("Before SMOTE:", y_train.value_counts().to_dict())
    print("After SMOTE:", pd.Series(y_train_smote).value_counts().to_dict())
else:
    X_train_smote, y_train_smote = X_train_scaled, y_train

## 8. Classification Models — Train, Predict, Evaluate All
We compare models using **Accuracy, Precision, Recall, F1-score** (not just accuracy, since it can be
misleading on imbalanced data), plus a confusion matrix for the best model.

In [ ]:
results = []
fitted_models = {}

def evaluate(name, model, X_tr, X_te, y_tr, y_te):
    model.fit(X_tr, y_tr)
    preds = model.predict(X_te)
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_te, preds),
        'Precision': precision_score(y_te, preds),
        'Recall': recall_score(y_te, preds),
        'F1': f1_score(y_te, preds)
    })
    fitted_models[name] = model
    return model

In [ ]:
# 1. Logistic Regression - baseline linear classifier, outputs probabilities via sigmoid function.
#    Fast, interpretable, works well when classes are roughly linearly separable.
evaluate('Logistic Regression', LogisticRegression(max_iter=1000), X_train_scaled, X_test_scaled, y_train, y_test)

# 2. KNN Classifier - classifies based on majority vote of 'k' nearest neighbors. Simple, no
#    training phase, but slow at prediction time and needs scaled features.
evaluate('KNN Classifier', KNeighborsClassifier(n_neighbors=5), X_train_scaled, X_test_scaled, y_train, y_test)

# 3. Decision Tree Classifier - splits data using if/else rules. Easy to interpret/visualize,
#    but prone to overfitting without depth limits.
evaluate('Decision Tree', DecisionTreeClassifier(max_depth=5, random_state=42), X_train, X_test, y_train, y_test)

# 4. Random Forest Classifier - bagging ensemble of decision trees, reduces overfitting/variance.
evaluate('Random Forest', RandomForestClassifier(n_estimators=200, random_state=42), X_train, X_test, y_train, y_test)

# 5. SVM (SVC) - finds the hyperplane that best separates classes with max margin. Strong with
#    clear margins of separation, needs scaled features.
evaluate('SVM (SVC)', SVC(kernel='rbf', probability=True, random_state=42), X_train_scaled, X_test_scaled, y_train, y_test)

# 6. Naive Bayes - probabilistic classifier based on Bayes' theorem, assumes feature independence.
#    Very fast, works surprisingly well as a baseline despite the "naive" independence assumption.
evaluate('Naive Bayes', GaussianNB(), X_train_scaled, X_test_scaled, y_train, y_test)

In [ ]:
# 7. AdaBoost - boosting ensemble: sequentially trains weak learners (shallow trees), giving more
#    weight to previously misclassified points.
evaluate('AdaBoost', AdaBoostClassifier(n_estimators=200, random_state=42), X_train, X_test, y_train, y_test)

# 8. Gradient Boosting - boosting ensemble that fits new trees to the RESIDUAL ERRORS of prior trees.
evaluate('Gradient Boosting', GradientBoostingClassifier(n_estimators=200, random_state=42), X_train, X_test, y_train, y_test)

# 9. XGBoost - optimized, regularized gradient boosting, generally faster & more accurate.
if XGB_AVAILABLE:
    evaluate('XGBoost', XGBClassifier(n_estimators=200, random_state=42, eval_metric='logloss'), X_train, X_test, y_train, y_test)

# 10. LightGBM - histogram-based gradient boosting, very fast on large data.
if LGBM_AVAILABLE:
    evaluate('LightGBM', LGBMClassifier(n_estimators=200, random_state=42, verbosity=-1), X_train, X_test, y_train, y_test)

In [ ]:
# 11. Voting Classifier - combines predictions from multiple DIFFERENT models by majority vote
#     (hard) or averaged probability (soft). Reduces variance by leveraging model diversity.
voting = VotingClassifier(estimators=[
    ('lr', LogisticRegression(max_iter=1000)),
    ('rf', RandomForestClassifier(n_estimators=200, random_state=42)),
    ('svc', SVC(probability=True, random_state=42))
], voting='soft')
evaluate('Voting Classifier (soft)', voting, X_train_scaled, X_test_scaled, y_train, y_test)

# 12. Stacking Classifier - trains a "meta-model" on top of base models' predictions, learning how
#     to best COMBINE them (smarter than simple voting).
stacking = StackingClassifier(estimators=[
    ('dt', DecisionTreeClassifier(max_depth=5, random_state=42)),
    ('knn', KNeighborsClassifier()),
    ('nb', GaussianNB())
], final_estimator=LogisticRegression())
evaluate('Stacking Classifier', stacking, X_train_scaled, X_test_scaled, y_train, y_test)

In [ ]:
results_df = pd.DataFrame(results).sort_values('F1', ascending=False).reset_index(drop=True)
results_df

## 9. Confusion Matrix & Classification Report (Best Model)

In [ ]:
best_name = results_df.iloc[0]['Model']
best_model = fitted_models[best_name]
X_te = X_test_scaled if best_name in ['Logistic Regression','KNN Classifier','SVM (SVC)','Naive Bayes',
                                        'Voting Classifier (soft)','Stacking Classifier'] else X_test
preds = best_model.predict(X_te)

cm = confusion_matrix(y_test, preds)
plt.figure(figsize=(4,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.title(f'Confusion Matrix - {best_name}')
plt.show()

print(classification_report(y_test, preds))

## 10. ROC-AUC Curve & Precision-Recall Curve
**Why ROC-AUC:** shows the tradeoff between True Positive Rate and False Positive Rate at every
threshold. AUC (Area Under Curve) close to 1.0 = excellent separability, 0.5 = random guessing.

**Why Precision-Recall curve:** more informative than ROC when classes are imbalanced, since it
focuses on the minority (positive) class performance directly.

In [ ]:
plt.figure(figsize=(7,6))
for name in ['Logistic Regression', 'Random Forest', 'SVM (SVC)', 'Gradient Boosting']:
    if name not in fitted_models:
        continue
    model = fitted_models[name]
    X_te = X_test_scaled if name in ['Logistic Regression', 'SVM (SVC)'] else X_test
    if hasattr(model, 'predict_proba'):
        probs = model.predict_proba(X_te)[:,1]
    else:
        probs = model.decision_function(X_te)
    fpr, tpr, _ = roc_curve(y_test, probs)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'{name} (AUC={roc_auc:.2f})')

plt.plot([0,1],[0,1],'k--', label='Random guess')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC Curves - Model Comparison')
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(7,6))
for name in ['Logistic Regression', 'Random Forest']:
    if name not in fitted_models:
        continue
    model = fitted_models[name]
    X_te = X_test_scaled if name == 'Logistic Regression' else X_test
    probs = model.predict_proba(X_te)[:,1]
    precision, recall, _ = precision_recall_curve(y_test, probs)
    plt.plot(recall, precision, label=name)

plt.xlabel('Recall'); plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend()
plt.show()

## 11. Multi-class Classification Note
Titanic is binary, but many real problems have 3+ classes (e.g. predicting `pclass`: 1st/2nd/3rd).
Two strategies extend binary classifiers to multi-class:
- **One-vs-Rest (OvR):** trains one binary classifier per class ("this class" vs "all others"), picks
  the class with highest confidence.
- **One-vs-One (OvO):** trains a classifier for every PAIR of classes, uses majority vote.

Scikit-learn handles this automatically for most models — just pass a multi-class target.

In [ ]:
# Quick multi-class example: predict pclass instead of survived
X_multi = data_encoded.drop(columns=['survived', 'pclass'])
y_multi = data_encoded['pclass']  # 1, 2, or 3

Xm_train, Xm_test, ym_train, ym_test = train_test_split(X_multi, y_multi, test_size=0.2, random_state=42, stratify=y_multi)

# OvR is the default strategy for LogisticRegression on multi-class problems
multi_clf = LogisticRegression(max_iter=1000, multi_class='ovr')
multi_clf.fit(Xm_train, ym_train)
print("Multi-class accuracy (predicting pclass):", accuracy_score(ym_test, multi_clf.predict(Xm_test)))
print(classification_report(ym_test, multi_clf.predict(Xm_test)))

## 12. Cross-Validation (Stratified)
**Why Stratified:** for classification, plain KFold might create folds with very different class
ratios. StratifiedKFold preserves class proportions in every fold -> more reliable evaluation.

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in [('Logistic Regression', LogisticRegression(max_iter=1000)),
                     ('Random Forest', RandomForestClassifier(n_estimators=200, random_state=42))]:
    X_use = X_train_scaled if name == 'Logistic Regression' else X_train
    scores = cross_val_score(model, X_use, y_train, cv=skf, scoring='f1')
    print(f"{name}: mean F1 = {scores.mean():.3f} (+/- {scores.std():.3f})")

## 13. Hyperparameter Tuning

In [ ]:
# GridSearchCV on Random Forest
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, None],
    'min_samples_split': [2, 5]
}
grid = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=3, scoring='f1', n_jobs=-1)
grid.fit(X_train, y_train)
print("Best params:", grid.best_params_, "| Best CV F1:", grid.best_score_)

In [ ]:
# RandomizedSearchCV on SVM
param_dist = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.01, 0.1],
    'kernel': ['rbf', 'linear']
}
rand_search = RandomizedSearchCV(SVC(probability=True, random_state=42), param_dist,
                                  n_iter=8, cv=3, scoring='f1', random_state=42, n_jobs=-1)
rand_search.fit(X_train_scaled, y_train)
print("Best params:", rand_search.best_params_, "| Best CV F1:", rand_search.best_score_)

## 14. Feature Importance & Selection

In [ ]:
rf = RandomForestClassifier(n_estimators=200, random_state=42).fit(X_train, y_train)
importances = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=False)

plt.figure(figsize=(6,4))
importances.plot(kind='bar')
plt.title('Random Forest Feature Importances')
plt.show()

In [ ]:
rfe = RFE(LogisticRegression(max_iter=1000), n_features_to_select=4)
rfe.fit(X_train_scaled, y_train)
print("Top 4 features (RFE):", list(X_train.columns[rfe.support_]))

## 15. Learning Curves

In [ ]:
train_sizes, train_scores, val_scores = learning_curve(
    RandomForestClassifier(n_estimators=200, random_state=42), X_train, y_train, cv=5,
    scoring='f1', train_sizes=np.linspace(0.1, 1.0, 5))

plt.figure(figsize=(6,4))
plt.plot(train_sizes, train_scores.mean(axis=1), 'o-', label='Training score')
plt.plot(train_sizes, val_scores.mean(axis=1), 'o-', label='Validation score')
plt.xlabel('Training examples'); plt.ylabel('F1 score')
plt.title('Learning Curve - Random Forest')
plt.legend()
plt.show()

## 16. Pipeline + Save Best Model

In [ ]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestClassifier(n_estimators=200, random_state=42))
])
pipeline.fit(X_train, y_train)
print("Pipeline F1 on test set:", f1_score(y_test, pipeline.predict(X_test)))

import joblib
joblib.dump(pipeline, 'best_classification_model.pkl')
# loaded = joblib.load('best_classification_model.pkl'); loaded.predict(new_data)
print("Model saved as best_classification_model.pkl")

## 17. Final Model Comparison Table

In [ ]:
results_df

## 18. Summary — Topics Covered in This Notebook
- EDA including class balance check
- Missing value imputation, feature engineering (family_size, is_alone)
- Encoding: OneHot / LabelEncoder
- Scaling: StandardScaler (and when NOT needed for trees)
- Stratified train-test split
- Handling class imbalance: SMOTE, class_weight
- Classification models: Logistic Regression, KNN, Decision Tree, Random Forest, SVM, Naive Bayes,
  AdaBoost, Gradient Boosting, XGBoost, LightGBM, Voting Classifier, Stacking Classifier
- Confusion matrix & classification report
- ROC-AUC curve, Precision-Recall curve
- Multi-class classification (OvR/OvO concept)
- Stratified K-Fold cross-validation
- Hyperparameter tuning: GridSearchCV, RandomizedSearchCV
- Feature importance & RFE
- Learning curves
- Pipelines + model persistence

**Next notebook:** `03_Unsupervised_Learning.ipynb`
